**Estimated time to read:** 10 minutes
**Depends on:** Data Ingestion, Precomputed Tracking & QC, JABS Demo
**Audience:** anyone with their own open-field-arena (OFA) recordings who wants to run them through this course's workflow, not just the example dataset

This page is not a notebook you run - it is a map. Everything else in this module works with
one curated example dataset. This page is for when you are ready to point the same pipeline at **your own mouse
video data**.


## Learning Goals

By the end of this page, you should be able to:

- Name the four stages of the official JABS pipeline (data acquisition, pose estimation,
  annotation and classifier training, genetic analysis) and which tool handles each
- Recognize what your own open-field-arena recordings need to look like to work with this pipeline
- Locate the Kumar Lab tools that turn raw video into JABS-compatible pose files
- Map your own data onto the same ingestion -> QC -> annotation -> classification -> analysis
  workflow this course walks through
- Know where to go for help if your data does not fit the expected format


## Is This Page For You?

This page is for you if:

- You have your own open-field-arena video of mice (or a similar single-animal, top-down
  recording setup)
- You want to annotate and classify a behavior using JABS, the way this course demonstrates
- You do not yet have pose-estimation keypoints for your video

If you already have JABS-format pose files (`*_pose_est_v6.h5`), skip ahead to
[Precomputed Tracking & QC](03_precomputed_tracking.html) - the same code runs on your files
wherever this course's pages point at `data/`.


## The Official JABS Pipeline

JABS is published as a four-stage, end-to-end pipeline, each stage handled by its own tool
(Choudhary, A., Geuther, B. Q., Sproule, T. J., Beane, G., Kohar, V., Trapszo, J., & Kumar, V., "JAX Animal Behavior System (JABS), a genetics-informed, end-to-end advanced
behavioral phenotyping platform for the laboratory mouse," eLife 107259 (2026),
[elifesciences.org/articles/107259](https://elifesciences.org/articles/107259)):

| Stage | Module | Tool |
|---|---|---|
| Data acquisition | JABS-DA | [JABS-data-pipeline](https://github.com/KumarLabJax/JABS-data-pipeline) |
| Pose estimation | - | [mouse-tracking-runtime](https://github.com/KumarLabJax/mouse-tracking-runtime) |
| Behavioral annotation & classifier training | JABS-AL | [JABS-behavior-classifier](https://github.com/KumarLabJax/JABS-behavior-classifier) |
| Genetic analysis | JABS-AI | (deploys trained classifiers across large genetic cohorts - not covered by this course) |

Steps 1-4 below walk you through the first three stages using your own data. The fourth stage,
JABS-AI, is about running a classifier across a large-scale genetic dataset to find genetic
drivers of behavior - a different scale of problem than this course's own
[Results and Analysis](05_results_and_analysis.html) page, which evaluates a single classifier
against ground truth. If you get as far as wanting to do genetics-scale analysis, JABS-AI is
where that work lives, but it is out of scope for this page.


## Step 1: Check Your Recording Setup

This is the **data acquisition** stage (JABS-DA) - the [JABS-data-pipeline](https://github.com/KumarLabJax/JABS-data-pipeline) repo has the hardware build guide if you are setting up a new rig from scratch.

**Arena, lighting, and camera** (Geuther, B.Q., Deats, S.P., Fox, K.J., Murray, S.A., Braun, R.E.,
White, J.K., Chesler, E.J., Lutz, C.M., Kumar, V. "Robust mouse tracking in complex environments
using neural networks." Communications Biology 2, 124 (2019).
[doi:10.1038/s42003-019-0362-1](https://doi.org/10.1038/s42003-019-0362-1)):

- 52 cm x 52 cm x 23 cm arena - white PVC plastic floor, gray PVC plastic walls, with a 2.54 cm
  chamfer on all inner edges (aids cleaning)
- LED ring light (F&V R300), calibrated to 600 lux
- Sentech STC-MB33USB camera with a Computar T3Z2910CS-IR lens, mounted about 100 cm above the
  arena, zoomed to 8 px/cm
- 640x480 px, 8-bit monochrome, 30 fps (cropped to 480x480 px for analysis)
- Single animal, top-down view only - the pipeline is not validated for multiple animals in
  frame
- A subset of arenas are set up for continuous 24-hour monitoring on a 12:12 light-dark cycle:
  250 lux during the day, under 5 lux at night, with additional 940 nm IR LED lighting for night
  recording

This same arena and the cropped 480x480 px @ 30 fps format are reused in later Kumar Lab
publications, e.g. Sheppard, K., Gardin, J., Sabnis, G.S., Peer, A., Darrell, M., Deats, S.,
Geuther, B., Lutz, C.M., Kumar, V. "Stride-level analysis of mouse open field behavior using
deep-learning-based pose estimation." Cell Reports 38, 110231 (2022).
[doi:10.1016/j.celrep.2021.110231](https://doi.org/10.1016/j.celrep.2021.110231).

Your video needs to reasonably match this setup for the pose model to produce usable keypoints;
a substantially different camera angle, arena size, or lighting will degrade pose quality in
ways the [Precomputed Tracking & QC](03_precomputed_tracking.html) page's checks are designed to
catch, not prevent.


## Step 2: Generate Pose Estimates from Your Video

This is the **pose estimation** stage. Your video needs to become a JABS-format pose file before anything else in this course applies.
The Kumar Lab's [mouse-tracking-runtime](https://github.com/KumarLabJax/mouse-tracking-runtime)
pipeline does exactly this: it runs a trained pose-estimation model on your video and writes out
a `poseest/` HDF5 file with `points` and `confidence` in the same 12-keypoint schema used
throughout this course.

- For the schema itself (what `points` and `confidence` mean, and why the keypoints are ordered
  the way they are), see [Data Ingestion](01_data_ingestion.html).
- For how a confidence map actually becomes a keypoint and a confidence value, see the
  [Pose Estimation deep dive](02_pose_estimation.html).

Follow the `mouse-tracking-runtime` repository's own setup instructions to run it against your
video - that pipeline, not this course, is where a pose model actually gets applied to new data.


## Step 3: Quality-check Your Pose Files

Once you have pose files for your own sessions, run them through the same QC this course applies
to its example dataset. [Precomputed Tracking & QC](03_precomputed_tracking.html) auto-detects
whichever `data/` subfolder you point it at (`data_subdir`) - drop your own pose files into a
folder there and the same thresholds, histograms, and per-keypoint reliability checks run
unchanged. Pay particular attention to the per-keypoint reliability chart: a keypoint that is
unreliable on your setup but fine on the course's example data is a sign your recording setup
differs from what the pose model expects.


## Step 4: Annotate, Train, and Classify

This is the **behavioral annotation and classifier training** stage - JABS-AL. With QC-passed sessions, follow [JABS Demo](live_demo.html) to open your data
in the JABS GUI, label a behavior, train a classifier, and run it - the same walkthrough this
course uses for its own example behavior (grooming) applies to whatever behavior you choose to
label in your data.


## Step 5: Analyze Your Results

Once you have classifier predictions on your own sessions,
[Results and Analysis](05_results_and_analysis.html) walks through turning them into
ethograms, bout-duration distributions, and classifier evaluation metrics - again, the same code,
run on your own predictions instead of the course's example set.


## Contact

If your data does not fit the setup above, or you run into trouble applying JABS to it, the
Kumar Lab maintains a support address for exactly this: **jabs@jax.org** (from the
[JABS-behavior-classifier README](https://github.com/KumarLabJax/JABS-behavior-classifier)).
That same README notes that multi-mouse support is still under development - contact them
directly if that is your use case.

The tool itself is developed and maintained at [The Jackson Laboratory](https://www.jax.org/)
by the [Kumar Lab](https://www.kumarlab.org/).


## Resources

- [JABS-data-pipeline](https://github.com/KumarLabJax/JABS-data-pipeline) - data acquisition (JABS-DA): hardware build guide and recording setup
- [mouse-tracking-runtime](https://github.com/KumarLabJax/mouse-tracking-runtime) - pose
  estimation on your own video
- [JABS-behavior-classifier](https://github.com/KumarLabJax/JABS-behavior-classifier) - the JABS
  GUI and CLI tools
- [JABS documentation](https://jabs-tutorial.readthedocs.io/en/latest/) - full JABS reference
- [Kumar Lab](https://www.kumarlab.org/) - lab homepage, publications, and protocols
- JABS-AI - the genetic-analysis stage (deploying classifiers across large genetic cohorts);
  see the JABS paper above for details - out of scope for this page


## What Next

This page only maps the path from your own video to a trained classifier - the actual pipeline
runs happen outside this course, since they depend on your own hardware, video, and behavior of
interest. If you get stuck at any single stage, the pages linked above go into far more depth on
that stage than this page does.
